# CS Count Summary

For each of the 4 fine-mapping settings (SuSiE vs SuSiE-SS × MAF1 vs MAF5),
report per-cohort:

- **Unique CS count** – number of distinct credible sets (identified by `event_ID × cs_index`)
- **Unique gene count with CS** – number of distinct `gene_ID`s that have at least one CS

Reported for three coverage thresholds: 95%, 70%, 50%.

| Setting | Method | MAF |
|---------|---------|-----|
| `std_maf5` | SuSiE (standard) | ≥5% |
| `std_maf1` | SuSiE (standard) | ≥1% |
| `ss_maf5`  | SuSiE-SS (summary stats) | ≥5% |
| `ss_maf1`  | SuSiE-SS (summary stats) | ≥1% |

In [3]:
suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(purrr)
  library(knitr)
})
options(datatable.showProgress = FALSE)

In [1]:
BASE <- '/mnt/lustre/lab/gwang/MiGA_eQTL_finemapping'

COMBINED_FILES <- c(
  std_maf5 = file.path(BASE, 'std_maf5_full/export/summary/combined.tsv.gz'),
  std_maf1 = file.path(BASE, 'std_maf1_full/export/summary/combined.tsv.gz'),
  ss_maf5  = file.path(BASE, 'ss_maf5_full/export/summary/combined.tsv.gz'),
  ss_maf1  = file.path(BASE, 'ss_maf1_full/export/summary/combined.tsv.gz')
)

CS_COLS <- c('cs_coverage_0.95', 'cs_coverage_0.7', 'cs_coverage_0.5')

KEEP_COLS <- c('gene_ID', 'event_ID', CS_COLS)

## Read all 4 combined files

In [4]:
dat <- rbindlist(
  imap(COMBINED_FILES, function(path, setting) {
    message(sprintf('Reading %s ...', setting))
    dt <- fread(path, select = KEEP_COLS, na.strings = c('', 'NA'))
    dt[, setting := setting]
    dt
  })
)

# Parse cohort as the 2nd '_'-delimited field of event_ID
# event_ID format: MiGA_<CellType>_eQTL_<GeneID>
dat[, cohort := str_split_i(event_ID, '_', 2)]

# Replace NA CS values with 0 (no credible set)
for (col in CS_COLS) set(dat, j = col, value = fifelse(is.na(dat[[col]]), 0L, dat[[col]]))

cat(sprintf('Total rows loaded: %s\n', format(nrow(dat), big.mark = ',')))
cat(sprintf('Settings: %s\n', paste(unique(dat$setting), collapse = ', ')))
cat(sprintf('Cohorts:  %s\n', paste(sort(unique(dat$cohort)), collapse = ', ')))

Reading std_maf5 ...

Reading std_maf1 ...

Reading ss_maf5 ...

Reading ss_maf1 ...



Total rows loaded: 6,947,266
Settings: std_maf5, std_maf1, ss_maf5, ss_maf1
Cohorts:  GFM, GTS, SVZ, THA


## Compute unique CS counts and unique gene counts

In [5]:
# For each CS coverage column:
#   n_unique_CS      = distinct (event_ID, cs_index) pairs where cs_index > 0
#   n_genes_with_CS  = distinct gene_IDs with at least one cs_index > 0
cs_summary <- rbindlist(lapply(CS_COLS, function(cs_col) {
  sub <- dat[get(cs_col) > 0L]
  sub[, cs_uid := paste0(event_ID, ':', get(cs_col))]
  sub[, .(n_unique_CS     = uniqueN(cs_uid),
           n_genes_with_CS = uniqueN(gene_ID)),
       by = .(setting, cohort)
  ][, coverage := cs_col]
}))

# Pretty coverage label
cs_summary[, coverage_label := fcase(
  coverage == 'cs_coverage_0.95', 'CS95',
  coverage == 'cs_coverage_0.7',  'CS70',
  coverage == 'cs_coverage_0.5',  'CS50'
)]

# Setting order
cs_summary[, setting := factor(setting, levels = c('std_maf5','std_maf1','ss_maf5','ss_maf1'))]

setorder(cs_summary, coverage_label, setting, cohort)

## Summary table: unique CS count

In [6]:
cs_count_wide <- dcast(
  cs_summary,
  coverage_label + setting ~ cohort,
  value.var = 'n_unique_CS'
)

cat('=== Unique CS count per setting × cohort ===\n')
kable(cs_count_wide, format = 'simple', big.mark = ',')

=== Unique CS count per setting × cohort ===




coverage_label   setting       GFM     GTS     SVZ     THA
---------------  ---------  ------  ------  ------  ------
CS50             std_maf5     7446   21582   37631   21534
CS50             std_maf1     8994   26014   45119   26158
CS50             ss_maf5      9040   27427   36594   28018
CS50             ss_maf1     11579   34058   42404   34848
CS70             std_maf5     6766   21018   37199   21000
CS70             std_maf1     8356   25535   44742   25661
CS70             ss_maf5      8894   27273   36437   27858
CS70             ss_maf1     11437   33916   42266   34684
CS95             std_maf5     5021   19090   35614   18974
CS95             std_maf1     6531   23735   43375   23859
CS95             ss_maf5      7964   26275   35413   26857
CS95             ss_maf1     10612   32946   41194   33648

## Summary table: unique gene count with CS

In [7]:
gene_count_wide <- dcast(
  cs_summary,
  coverage_label + setting ~ cohort,
  value.var = 'n_genes_with_CS'
)

cat('=== Unique gene count with CS per setting × cohort ===\n')
kable(gene_count_wide, format = 'simple', big.mark = ',')

=== Unique gene count with CS per setting × cohort ===




coverage_label   setting      GFM     GTS     SVZ     THA
---------------  ---------  -----  ------  ------  ------
CS50             std_maf5    5502    9816   11747    9968
CS50             std_maf1    6331   10846   12674   10985
CS50             ss_maf5     5697    9477   11185    9604
CS50             ss_maf1     6718   10658   12260   10787
CS70             std_maf5    4986    9518   11567    9680
CS70             std_maf1    5874   10597   12539   10734
CS70             ss_maf5     5588    9430   11170    9577
CS70             ss_maf1     6624   10637   12254   10774
CS95             std_maf5    3680    8529   10952    8636
CS95             std_maf1    4573    9738   12094    9887
CS95             ss_maf5     4945    9187   11044    9322
CS95             ss_maf1     6122   10505   12199   10654

## Long-format combined table (for export)

In [8]:
export_tbl <- cs_summary[, .(coverage_label, setting, cohort, n_unique_CS, n_genes_with_CS)]
setorder(export_tbl, coverage_label, setting, cohort)

print(export_tbl)

fwrite(export_tbl, 'cs_count_summary.tsv', sep = '\t')
cat('\nSaved: cs_count_summary.tsv\n')

    coverage_label  setting cohort n_unique_CS n_genes_with_CS
            <char>   <fctr> <char>       <int>           <int>
 1:           CS50 std_maf5    GFM        7446            5502
 2:           CS50 std_maf5    GTS       21582            9816
 3:           CS50 std_maf5    SVZ       37631           11747
 4:           CS50 std_maf5    THA       21534            9968
 5:           CS50 std_maf1    GFM        8994            6331
 6:           CS50 std_maf1    GTS       26014           10846
 7:           CS50 std_maf1    SVZ       45119           12674
 8:           CS50 std_maf1    THA       26158           10985
 9:           CS50  ss_maf5    GFM        9040            5697
10:           CS50  ss_maf5    GTS       27427            9477
11:           CS50  ss_maf5    SVZ       36594           11185
12:           CS50  ss_maf5    THA       28018            9604
13:           CS50  ss_maf1    GFM       11579            6718
14:           CS50  ss_maf1    GTS       34058         